# Phân Tích Các Nguyên Tắc Chơi Của Tác Tử Cấp 2 (`Level2HeuristicAgent`)

Notebook này mô tả chính xác và trực quan toàn bộ hệ thống **luật phân tầng 1-ply**, **hàm lượng giá vị trí**, **Static Exchange Evaluation (SEE)** và **cơ chế giải quyết xung đột luật** đang được cài đặt trong `chess_ai/agents/level2_heuristic.py` và `chess_ai/core/evaluation.py`.

### Kiến trúc phân tầng ưu tiên 6 Tiers (`Tier 1` $\rightarrow$ `Tier 6`)
Trong `Level2HeuristicAgent.select_move()`, mỗi nước đi hợp lệ được đánh giá ở độ sâu 1-ply và phân vào đúng một **tầng ưu tiên rạch ròi (`tier` nhỏ hơn = ưu tiên chọn cao hơn)**:

| Tầng (`Tier`) | Mã luật (`selected_rule`) | Điều kiện kích hoạt & Ý nghĩa |
| :---: | :--- | :--- |
| **Tier 1** | `RULE_T1_MATE_IN_1` | **Thắng ngay lập tức:** Nước đi dẫn tới `board.is_checkmate()`. Trả về ngay không cần xét các nước khác. |
| **Tier 2** | `RULE_T2_SAFE_PROMOTION` / `RULE_T2_WINNING_CAPTURE_SEE` | **Phong Hậu an toàn & Bắt quân lời ròng:** `promotion == QUEEN` (`opp_see_threat < 800`) hoặc `see_capture(move) > 0` với `opp_see_threat < move_see` (không vi phạm Tier 4–6). |
| **Tier 3** | `RULE_T3_SAFE_CHECK` / `RULE_T3_POSITIONAL` | **Kiểm soát vị trí & Chiếu an toàn:** Tối ưu `evaluate_board() + center_control_weight * center_control_score() - see_hanging_weight * opp_see_threat` (+`30.0 cp` nếu chiếu an toàn `opp_see_threat < 100`). |
| **Tier 4** | `RULE_T4_AVOID_REPETITION_2/3` | **Phạt lặp thế cờ:** Khi `static_eval >= repetition_eval_threshold` (`0.0`) và lặp 2/3 lần; chỉ chọn nếu hết nước an toàn ở Tier 1–3. |
| **Tier 5** | `RULE_T5_AVOID_STALEMATE` | **Chống hòa Pat / thiếu lực lượng:** Khi `root_eval >= 100.0` mà nước đi gây `stalemate` hoặc `insufficient_material`. |
| **Tier 6** | `RULE_T6_AVOID_MATE_IN_1` | **Cấm nước tự sát:** Kiểm tra **đầu tiên** sau Tier 1; nếu đi xong đối thủ có `Mate-in-1`, dừng xét thưởng và đẩy xuống đáy (`-MATE_VALUE + 100`). |

> **Lưu ý đọc bảng:** `Tier 4`, `Tier 5`, `Tier 6` là **nhóm nước vi phạm** bị đẩy xuống cuối bảng xếp hạng. Chỉ cần tồn tại ít nhất một nước an toàn ở `Tier 1–3`, mọi nước ở `Tier 4–6` đều bị loại bỏ hoàn toàn.

In [ ]:
from collections import Counter
from pathlib import Path
import sys

import chess
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from chess_ai.agents.level1_random import Level1RandomAgent
from chess_ai.agents.level2_heuristic import (
    Level2HeuristicAgent,
    center_control_score,
    see_capture,
)
from chess_ai.config import load_yaml_config
from chess_ai.core.chess_env import ChessEnv
from chess_ai.core.evaluation import (
    bishop_pair_score,
    evaluate_board,
    pawn_structure_score,
    rook_file_score,
)
from chess_ai.ui.board_renderer import render_board_image
from chess_ai.utils.epd import load_epd_file

cfg = load_yaml_config("search_params.yaml")["heuristic_level2"]
print("Cấu hình heuristic_level2 từ configs/search_params.yaml:")
for k, v in cfg.items():
    print(f"  - {k}: {v}")

## 1. Nguyên tắc Kiểm soát Vị trí (`Tier 3` — `evaluate_board` + `center_control_score`)

Trong tầng định vị an toàn (**Tier 3**), Level 2 đánh giá chất lượng thế trận sau 1 nước đi qua 2 lớp:
1. **Lượng giá lõi (`evaluate_board` — `chess_ai/core/evaluation.py`):**
   - **Material Balance:** `P=100, N=320, B=330, R=500, Q=900, K=20000`.
   - **PST Michniewski:** Bảng vị trí 64 ô cho từng loại quân, riêng Vua nội suy theo `phase` giữa `K_middlegame` và `K_endgame`.
   - **Mobility & King Safety:** Thưởng độ cơ động (`N×4, B×4, R×2, Q×1`) và lá chắn Tốt trước Vua (`+12` / `+6`), phạt cột mở quanh Vua (`-15` / `-10`) theo `phase`.
   - **Cấu trúc Tốt (`pawn_structure_score`):** Thưởng **Tốt thông** (`+10` đến `+95 cp` theo hàng tiến triển), phạt **Tốt chồng** (`-15 cp`/Tốt dư) và **Tốt cô lập** (`-15 cp`).
   - **Xe trên cột mở / nửa mở (`rook_file_score`):** Thưởng `+20 cp` (cột mở hoàn toàn không có Tốt) và `+10 cp` (cột nửa mở không có Tốt ta).
   - **Cặp Tượng (`bishop_pair_score`):** Thưởng `+35 cp` khi sở hữu $\ge 2$ Tượng.
2. **Kiểm soát 4 ô trung tâm (`center_control_score`):**
   - Đếm hiệu số quân tấn công (`+1`/`-1`) và chiếm đóng (`+2`/`-2`) trên 4 ô **`d4, d5, e4, e5`**, nhân trọng số `center_control_weight` (`8.0`).

In [ ]:
positions = {
    "Thế cờ xuất phát (đối xứng)": chess.Board(),
    "Trắng chiếm trung tâm (1. e4 e5 2. Nf3 Nc6 3. d4)": chess.Board(
        "r1bqkbnr/pppp1ppp/2n5/4p3/3PP3/5N2/PPP2PPP/RNBQKB1R b KQkq - 0 3"
    ),
    "Trắng có Tốt thông e6 vs Đen Tốt chồng+cô lập a7/a6": chess.Board(
        "4k3/p7/p3P3/8/8/8/8/4K3 w - - 0 1"
    ),
    "Trắng Xe cột mở e1 + Cặp Tượng vs Đen Xe cột nửa mở d8": chess.Board(
        "3rk3/8/8/8/8/8/3P4/2B1RKB1 w - - 0 1"
    ),
}

rows = []
for name, b in positions.items():
    eval_w = evaluate_board(b) if b.turn == chess.WHITE else -evaluate_board(b)
    cc_w = center_control_score(b, chess.WHITE)
    pawn_diff = pawn_structure_score(b, chess.WHITE) - pawn_structure_score(b, chess.BLACK)
    rook_diff = rook_file_score(b, chess.WHITE) - rook_file_score(b, chess.BLACK)
    bp_diff = bishop_pair_score(b, chess.WHITE) - bishop_pair_score(b, chess.BLACK)
    rows.append({
        "Thế cờ": name,
        "Static Eval (W cp)": round(eval_w, 2),
        "Pawn Struct Δ": pawn_diff,
        "Rook File Δ": rook_diff,
        "Bishop Pair Δ": bp_diff,
        "Center Ctrl (W)": cc_w,
        "Center Bonus (cp)": round(cc_w * cfg["center_control_weight"], 2),
    })

display(pd.DataFrame(rows))

# Hiển thị trực quan 4 thế cờ (tái sử dụng bộ quân & bảng màu Pygame UI, tô vàng 4 ô trung tâm d4/d5/e4/e5)
fig, axes = plt.subplots(1, len(positions), figsize=(14, 3.8), dpi=120)
center_sqs = (chess.D4, chess.D5, chess.E4, chess.E5)
for ax, (name, b) in zip(axes, positions.items()):
    ax.imshow(render_board_image(b, highlight_squares=center_sqs, board_size=320))
    ax.set_title(name, fontsize=8.5)
    ax.axis("off")
fig.tight_layout()
plt.show()

## 2. Nguyên tắc Trao đổi Quân (`see_capture` — Static Exchange Evaluation)

Hàm `see_capture(board, move)` mô phỏng chuỗi ăn qua lại trên cùng ô đích theo thứ tự **Least Valuable Attacker** (`P -> N -> B -> R -> Q -> K`) để phân biệt chính xác:
- **Bắt quân lời (`SEE > 0`):** Ăn quân không bảo vệ hoặc nhiều quân đánh ít quân hơn.
- **Đổi quân ngang (`SEE == 0`):** Đổi ngang giá trị.
- **Thí quân lỗ (`SEE < 0`):** Ví dụ Hậu (`900`) ăn Tốt (`100`) có Tốt khác bảo vệ (`SEE = -800`).

Ngoài ra, sau khi đi thử `move`, Level 2 gọi `_max_opponent_see_gain(board)` để phát hiện đe dọa ăn quân có lợi nhất của đối thủ ở lượt sau và trừ vào điểm theo hệ số `see_hanging_weight` (`0.90`).

In [ ]:
see_cases = [
    (
        "Hậu ăn Tốt có Tốt khác bảo vệ (Qd1xd5 vs e6)",
        "4k3/8/4p3/3p4/8/8/8/3QK3 w - - 0 1",
        "d1d5",
    ),
    (
        "2 quân Trắng (Mã+Xe) vs Mã Đen trên d5 được Tốt e6 bảo vệ",
        "4k3/8/4p3/3n4/8/2N5/8/3RK3 w - - 0 1",
        "c3d5",
    ),
    (
        "Tốt Trắng ăn Mã Đen không có bảo vệ (e4xd5)",
        "4k3/8/8/3n4/4P3/8/8/4K3 w - - 0 1",
        "e4d5",
    ),
]

see_rows = []
for label, fen, uci in see_cases:
    b = chess.Board(fen)
    m = chess.Move.from_uci(uci)
    val = see_capture(b, m)
    see_rows.append({
        "Tình huống": label,
        "Nước đi": uci,
        "SEE (cp)": val,
        "Đánh giá": "Lời" if val > 0 else ("Hòa" if val == 0 else "Lỗ"),
    })

display(pd.DataFrame(see_rows))

fig, axes = plt.subplots(1, len(see_cases), figsize=(11.5, 3.8), dpi=120)
for ax, (label, fen, uci) in zip(axes, see_cases):
    b = chess.Board(fen)
    m = chess.Move.from_uci(uci)
    val = see_capture(b, m)
    img = render_board_image(b, move=m if val >= 0 else None, bad_move=m if val < 0 else None, board_size=320)
    ax.imshow(img)
    ax.set_title(f"{label}\n(SEE = {val:+d} cp)", fontsize=8.5)
    ax.axis("off")
fig.tight_layout()
plt.show()

## 3. Thứ tự Ưu tiên & Giải quyết Xung đột Luật (`Rule Conflicts`)

Khi một nước đi đồng thời kích hoạt tín hiệu dương (`WINNING_CAPTURE`, `SAFE_CHECK`, `IMPROVES_POSITION`) và tín hiệu âm (`ALLOWS_OPP_MATE_IN_1`, `CAUSES_STALEMATE_WHILE_WINNING`, `CAUSES_REPETITION`, `LEAVES_PIECE_HANGING`), `Level2HeuristicAgent` ghi nhận xung đột vào `last_search_stats["conflicts_detected"]` và `last_search_stats["conflict_examples"]`.

Dưới đây là 4 kịch bản kiểm chứng:
1. **Mate-in-1 (`Tier 1`):** Chiếu hết ngay, bỏ qua mọi luật khác.
2. **Bẫy ăn Hậu vs Để hở Mate-in-1 (`Tier 6`):** Trắng có thể ăn không Hậu Đen (`Qxa2`), nhưng đi xong sẽ bị `...Re1#` chiếu hết hàng cuối $\rightarrow$ `Qxa2` bị đẩy xuống `Tier 6` (`RULE_T6_AVOID_MATE_IN_1`) để chọn nước an toàn ở `Tier 3`.
3. **Chống hòa Pat (`Tier 5`):** Đang thắng lớn ở tàn cuộc, loại bỏ các nước gây `stalemate`.
4. **Trung cuộc hỗn hợp:** Xung đột giữa cải thiện vị trí / bắt quân và treo quân.

In [ ]:
agent = Level2HeuristicAgent(seed=42)

conflict_scenarios = [
    (
        "1. Mate-in-1 (Scholar's Mate)",
        "r1bqkb1r/pppp1ppp/2n2n2/4p2Q/2B1P3/8/PPPP1PPP/RNB1K1NR w KQkq - 4 4",
    ),
    (
        "2. Ăn Hậu a2 (+900) vs Bị ...Re1# (Mate-in-1)",
        "4r1k1/5ppp/8/8/8/8/q4PPP/3Q2K1 w - - 0 1",
    ),
    (
        "3. Tàn cuộc thắng lớn: Tránh hòa Pat (Anti-Stalemate)",
        "k7/8/1Q6/2K5/8/8/8/8 w - - 0 1",
    ),
    (
        "4. Trung cuộc có xung đột vị trí vs treo quân",
        "r1bqkb1r/pppp1ppp/2n2n2/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4",
    ),
]

scenario_rows = []
scenario_results = []
for title, fen in conflict_scenarios:
    env = ChessEnv(fen=fen)
    move = agent.select_move(env)
    st = dict(agent.last_search_stats)
    scenario_results.append((title, env.board.copy(stack=False), move, st))
    scenario_rows.append({
        "Kịch bản": title,
        "Nước chọn": move.uci(),
        "Tier": st["selected_tier"],
        "Luật quyết định": st["selected_rule"],
        "Số xung đột": st["conflicts_detected"],
        "Ví dụ xung đột": "; ".join(st["conflict_examples"][:2]),
    })

pd.set_option("display.max_colwidth", 100)
display(pd.DataFrame(scenario_rows))

# Vẽ trực quan 4 kịch bản xung đột luật (mũi tên xanh: nước Level 2 chọn | mũi tên đỏ: nước bẫy bị loại)
fig, axes = plt.subplots(1, len(scenario_results), figsize=(14, 4.0), dpi=120)
trap_moves = {1: chess.Move.from_uci("d1a2")}  # Kịch bản 2: ăn Hậu a2 bị ...Re1#
for idx, (ax, (title, b_vis, chosen, st)) in enumerate(zip(axes, scenario_results)):
    img = render_board_image(b_vis, move=chosen, bad_move=trap_moves.get(idx), board_size=320)
    ax.imshow(img)
    ax.set_title(f"{title}\nChọn: {chosen.uci()} | Tier {st['selected_tier']}", fontsize=8.5)
    ax.axis("off")
fig.tight_layout()
plt.show()

## 4. Thống kê Tần suất Kích hoạt Luật & Xung đột khi Thi đấu (`Level 2` vs `Level 1`)

Chạy nhanh 10 ván (5 thế cờ khai cuộc cân bằng từ `resources/openings_50.epd` $\times 2$ màu) giữa `Level2HeuristicAgent` và `Level1RandomAgent` để quan sát phân bố tầng ưu tiên (`Tier`), mã luật quyết định (`selected_rule`) và tỷ lệ nước đi có xung đột luật.

In [ ]:
openings = load_epd_file(PROJECT_ROOT / "resources" / "openings_50.epd")[:5]
l2 = Level2HeuristicAgent(seed=42)
l1 = Level1RandomAgent(seed=42)

rule_counts = Counter()
tier_counts = Counter()
total_l2_moves = 0
moves_with_conflicts = 0
l2_score = 0.0
games = 0

for op in openings:
    for l2_is_white in (True, False):
        env = ChessEnv(fen=op.fen)
        l2.reset()
        l1.reset()
        plies = 0
        while not env.is_terminal() and plies < 160:
            is_w_turn = env.current_player() == 1
            if is_w_turn == l2_is_white:
                m = l2.select_move(env)
                st = l2.last_search_stats
                total_l2_moves += 1
                rule_counts[st["selected_rule"]] += 1
                tier_counts[st["selected_tier"]] += 1
                if st["conflicts_detected"] > 0:
                    moves_with_conflicts += 1
            else:
                m = l1.select_move(env)
            env.step(m)
            plies += 1

        res = env.result()
        if res is None or res == 0.0:
            l2_score += 0.5
        elif (res == 1.0 and l2_is_white) or (res == -1.0 and not l2_is_white):
            l2_score += 1.0
        games += 1

print(f"Kết quả Level 2 vs Level 1 ({games} ván): điểm quy đổi S = {l2_score / games:.2f}")
print(f"Tổng nước đi Level 2: {total_l2_moves} | Nước có xung đột luật: {moves_with_conflicts} ({moves_with_conflicts / max(1, total_l2_moves) * 100:.1f}%)")
print("Phân bố theo Tier:", dict(sorted(tier_counts.items())))

df_rules = pd.DataFrame(
    [{"Luật": k, "Số lần": v, "Tỷ lệ (%)": round(v / total_l2_moves * 100, 2)} for k, v in rule_counts.most_common()]
)
display(df_rules)

fig, ax = plt.subplots(figsize=(8, 3.6), dpi=120)
ax.barh(df_rules["Luật"][::-1], df_rules["Số lần"][::-1], color="#3B82F6")
ax.set_xlabel("Số lần được chọn")
ax.set_title("Phân bố luật quyết định của Level 2 khi đấu Level 1 (10 ván)")
ax.grid(axis="x", linestyle="--", alpha=0.4)
fig.tight_layout()
plt.show()